# Parte 3 — Experimento A: DistilBERT
### Workshop: Clasificación de Emociones en Twitter

**Modelo:** `distilbert-base-uncased`  
**Pre-entrenamiento:** Wikipedia + BookCorpus (texto formal)  
**Tokenizador:** WordPiece

Este es nuestro **baseline**. Pre-entrenado en texto formal, sin conocimiento específico del lenguaje de Twitter.

**Prerequisito:** haber ejecutado `part-1-data.ipynb` y `part-2-pipeline.ipynb`

In [ ]:
# Ejecutar primero part-2-pipeline.ipynb para tener todas las funciones disponibles
%run part-2-pipeline.ipynb

## Configuración del experimento

In [ ]:
MODEL_CHECKPOINT = "distilbert-base-uncased"
HF_REPO          = "your-username/tweeteval-emotion-distilbert"  # <-- cambia esto
LR               = 2e-5

### 📝 TODO 3.1 — Tokenizar el dataset con DistilBERT

Usa `make_tokenized_dataset` con el tokenizador correcto y guarda el resultado en `ds_distilbert`.

In [ ]:
# TODO 3.1 ── Tokenizar con DistilBERT
# ─────────────────────────────────────────────────────────────────────────────
tok_distilbert = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)
ds_distilbert  = make_tokenized_dataset(tok_distilbert)

print(ds_distilbert)

### 📝 TODO 3.2 — Cargar el modelo y contar parámetros

In [ ]:
# TODO 3.2 ── Cargar DistilBERT para clasificación
# ─────────────────────────────────────────────────────────────────────────────
model_distilbert = AutoModelForSequenceClassification.from_pretrained(
    MODEL_CHECKPOINT,
    num_labels=NUM_LABELS,
    id2label=ID2LABEL,
    label2id=LABEL2ID,
)

total     = sum(p.numel() for p in model_distilbert.parameters())
trainable = sum(p.numel() for p in model_distilbert.parameters() if p.requires_grad)
print(f"Parámetros totales:     {total:,}")
print(f"Parámetros entrenables: {trainable:,}")

### 📝 TODO 3.3 — Entrenar y evaluar

In [ ]:
# TODO 3.3 ── Entrenamiento de DistilBERT
# ─────────────────────────────────────────────────────────────────────────────
trainer_distilbert = make_trainer(
    model_distilbert,
    tok_distilbert,
    ds_distilbert,
    output_dir="./checkpoints/distilbert",
    lr=LR,
)

trainer_distilbert.train()

plot_training_curves(trainer_distilbert, title="DistilBERT — TweetEval Emotion")

metrics_distilbert = full_evaluation(
    trainer_distilbert,
    ds_distilbert["test"],
    model_name="DistilBERT",
)
print(metrics_distilbert)

## Push to Hub

In [ ]:
# Requiere: huggingface-cli login  (o haber cargado el token con python-dotenv)
trainer_distilbert.push_to_hub(
    HF_REPO,
    commit_message="DistilBERT fine-tuning — TweetEval emotion",
)
print(f"Modelo publicado en: https://huggingface.co/{HF_REPO}")